# M50 --- CirCor DigiScope as a **far-OOD negative control**

> **This is not a transfer test, and it must never be presented as one.**
> CirCor is a phonocardiogram corpus: it holds **heart** sounds, labelled murmur present /
> absent / unknown. A crackle/wheeze classifier has zero label overlap with that, so scoring
> it with the ICBHI metric would produce no generalization number at all. Cross-dataset
> transfer lives in `M49_cross_dataset/`.

**No CirCor label is read here.** Not the murmur annotation, not the segmentation `.tsv`, not
the demographics. The corpus enters as one thing only: audio that is definitionally not a
respiratory cycle. That is what makes this valid rather than a category error.

## The two questions

| | Question | How |
|---|---|---|
| **Rejection** | can the frozen model tell this is not its domain? | post-hoc energy score `-logsumexp(z)`, ICBHI-known vs CirCor-unknown, AUROC with a **patient-level** bootstrap |
| **Abstention** | when it is wrong, does it know? | max-softmax, entropy and the predicted-class histogram on heart sounds, against the same on ICBHI |

M49 found the model **wrong and confident** on SPRSound --- mean max-softmax 0.9435 at 0.2479
accuracy. This is the extreme point of that curve.

## The caveat that travels with the number

CirCor is **far-OOD**: a different organ. The paper's existing open-set arm reports
**AUROC 0.6466** on **near-OOD** unknowns --- held-out lung disease classes, **19 patients**.
Rejecting a heartbeat is an *easier* task than rejecting an unseen lung pathology.

So this number **does not replace, upgrade or supersede 0.6466.** It is a second, easier point
on a difficulty axis, and it earns its place by showing the detector functions at all --- which
n=19 could not. Both numbers get reported with their unknown sets named. The sentence ships
inside the results JSON as `openset.difficulty_note` so it cannot be lost in transcription.

## Before you press Run All

| | |
|---|---|
| **Accelerator** | GPU T4 |
| **Internet** | not needed --- everything is attached |
| **Add Data** | `bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2` |
| **Add Data** | `vbookshelf/respiratory-sound-database` (ICBHI --- the known side *and* the gate) |
| **Add Data** | the checkpoint: `Asif's/M22_v2/Results/best_model.pth` as a private dataset |
| **Runtime** | roughly 15 min |

### Bring back

`results_M50_circor.json`, `logits_M50_*.npy`, `M50_openset.png` --- zipped in the last cell.
Commit them into `M50_negative_control/`.


## 1. Environment

In [ ]:
!pip -q install librosa soundfile
import glob, json, os, sys, time
import numpy as np
print(sys.version)
import torch, librosa
print("torch", torch.__version__, "| librosa", librosa.__version__, "| cuda:",
      torch.cuda.is_available())

WORK = "/kaggle/working/M50"
os.makedirs(WORK, exist_ok=True)

# Set to a row count for a two-minute wiring check; None for the real thing.
SMOKE = None
# Cap the windows taken from any one recording. None keeps them all; the bootstrap groups by
# patient either way, so a long recording cannot dominate the interval.
MAX_WINDOWS = None


## 2. Locate the checkpoint, ICBHI and CirCor

By glob, not by hard-coded path --- Kaggle's dataset mirrors differ in layout.

In [ ]:
def find_one(pattern, what, hint=""):
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"could not find {what} with {pattern!r}. {hint}")
    return hits[0]

# The paper's best model. NOT best_model_official.pth (same folder, 0.5641, published split
# verbatim, leaks patients 156 and 218) -- m49_xval.load_checkpoint reads split_method out of
# the cfg and raises on it, so the wrong file cannot be used by accident.
CKPT_NAME = "best_model.pth"
CKPT = find_one(f"/kaggle/input/**/{CKPT_NAME}", "the checkpoint",
                f"Upload Asif's/M22_v2/Results/{CKPT_NAME} as a private Kaggle dataset.")
ICBHI_AUDIO = os.path.dirname(find_one(
    "/kaggle/input/**/audio_and_txt_files/*.wav", "the ICBHI audio",
    "Add Data -> vbookshelf/respiratory-sound-database."))
ICBHI_SPLIT = find_one("/kaggle/input/**/ICBHI_challenge_train_test.txt",
                       "the official ICBHI split file",
                       "It ships with the same dataset. A notebook that cannot find it must "
                       "raise, never fall back (Model_Training_Protocol.md 1.1).")

# CirCor: the Kaggle mirror nests the wavs under training_data/. Anchor on a file pattern that
# only CirCor has, then take its directory -- a hard-coded path that silently resolves to an
# empty directory is a failure mode this project has already been bitten by.
CIRCOR_ROOT = os.path.dirname(find_one(
    "/kaggle/input/**/training_data/*.wav", "the CirCor audio",
    "Add Data -> bjoernjostein/the-circor-digiscope-phonocardiogram-dataset-v2"))
print("checkpoint :", CKPT)
print("ICBHI audio:", ICBHI_AUDIO, len(glob.glob(ICBHI_AUDIO + "/*.wav")), "wavs")
print("CirCor     :", CIRCOR_ROOT, len(glob.glob(CIRCOR_ROOT + "/*.wav")), "wavs")


## 3. The three modules

`m45_ablation.py` supplies the mel parameters and the corrected-split loader; `m49_xval.py`
supplies the checkpoint loader, the forward pass and the ICBHI gate; `m50_openset.py` adds
the CirCor index and the open-set scorers. Nothing is re-implemented --- a second copy of the
spectrogram code would make every number here a measurement of the gap between two scripts.

All three are carried as gzip+base64 and decoded byte-identical to the repo, so the cells
below look like blobs rather than source. That is deliberate: a quoted literal that ends one
character early runs the rest of the module as *cell* code.

In [ ]:
M45_ABLATION_B64 = (
'''H4sIAAAAAAAC/7Vde3PbOJL/P1X5Dlim5kwmEiPJj02c1Vw5iTPj2jx8tmdn9zQqhpIgi2tK5JCU
bY3XW/ch7hPeJ7lfNwASpORHZu9cE5svNBqNRqNf6HEc5+mTTzu74n/+67/Fu2SeJgu5KMQLkWYy
zZKxzPNocS7CURwWUbIQ+K+YSTGSeSHmyUTGwj05+7PI5K/LKJNztM1FtyMOPr8XXe/pk6dP3h6c
Hn48+ny4Lz71esFljzv6lIyiWH6WxV966Oo0leOD5Tk1bolkOo3GURiLo3dvfzwSHX93r9MTA/zt
vOq2cL/X3f7jEHg8ffLVfBvsdYKdTpACRcAIosVEphK/cD1OskyOCzn56osvCykuwyzCYKQYz8LF
ucxFKjORJVdvhLyU2aqYYbRPn8g4lyLKxUzGExEWPGSNPqBFqfRpaD//+Dfx5fOhOH13cnR8Jj58
ORFnP38RJ4f/8dPRyeGnw89np0+fCPyAQoFNIX8+ETmwIoKCWHm4ykWeiElEqMarfeF8ktk5usS8
hIsJ/u6JaFEkGLUsp0KBNmAiNS+YtCQr3tD1SoSZ5Id5OJfiAlQBcTHMCIQZS9+hIZz9eCjOfjw5
PBSfjk5Pjz7/IE7PDn44PBUHJ4fi85efxcmXn09b4u1PZ/js8G/i4P17cXKASwz1x4PPGOunL385
LEdZorM9EnGUgxfy6LrBSiB27ouzWSYl4QME52pMQLQIaUqKGUg+iSZikRQKtLwGMDVGzEqKCYij
hdwHcfAqjcEEhRiBUu00zHMxjeJCZi2BcSYR9dliKmKm2+PVOJYKZjhHw2I5kegmm4dAl8nqi1Mz
hlCE+QXAJRn6TfIKvwT8L66yqCjkAr1lecH8QFDPDOEXyZWI0AXPuJwI9yshSPh9bYmvCjVJl8CD
EPhqpnCcLKbRuadwNvOogH/58IG+oiVFFASVQA2swWgkM9APz5bZgpfrYlXSVjEHtRFXyRL8rDhf
HHS4C01gYn4CVYQlmQteJ7iYyzBfZhhDeB5GC0zEQYdXcYjZjUlajMKcJ8SA5n4U4GhxCdJOaHaZ
FdHRgllAhHEmw8lKZMuFGapIQ1qOquPREtMoplky53GUFD5NxHG3fbzNVCZ+JGgt4hXNjfxgX8hw
PBMFCJLzsuG5I2JQX2qZ5OKKaaiZbLRcgTFPgBR1GGU2OdAyTZM8wjDUKiUGAR8cdNsHewzyeKd9
vOuLA8GfRZdSNVeguVuDLJE0JDmSzMEbk9DACidJqkQN4/oGHy3keciwaARoZhYvLxEFuvye3hMR
5pFaZ5py1noRI3AFJjqPzmeYtGVBD+YSfMzEPaDFTstcnB4fvhPvj95DApyJg9M/k2hrEfqH7yHw
xOFfjt4ffn6nl/3xTvCbzJI0nPgQVDsWtxRFFqEXGh6YOscSBu5g53GW5DkjfpWFKdbsZCJ5TmQR
0deYsKlmS5pDXrNM42myxG+s9RhUdNERWKnf8TuvX70BheKV2O59R3zPLUDkEUbLX+x6vjhkFudd
RQFXcEGffAZmIL4D9zJar0SOaU9BIHQyifC05+/0RI5FCTHNjGpIWkTUE1D65/a1uIqKmYL9dZH6
9OqrXiizMJsQEcZYCAs948R7sQwvCVNwVY7XRbRYRsWKthy1InMZVpz/DghHY0H0ogkGyzCiOc0/
+Ag9pFkI1hwzn/NINcJRbrgDY4kmUZKvFmOIjGisSTFL0OoN84QG355kEa1VJs54yagq4cLsO0nG
Sy3ZpmEUY8JZHaDVEi7RgYYbQ95F2KR5yEqk5Ql2o3Q5gsCdoXUpI0M19UQ18fPR2Y/oZBoyxJcJ
oUUbL8TMCuPWAvwyiSZMIYiCQk4xdN5YgB+tskLmvMIl5ivT0kURLh9DnvDMTxK6v0qyC/pAgx0n
yxyUYb4geQlZz9/54ogFLksDTD11Mk+wPEOiApZSaHifhiNA3plhqtFKwwal5fiCtkaeVdpd1RuI
1EqMWj99o3e4lrxsYbEscznxdNuuEG1bi6rablPTe9v2qO3RHAIE6hiIHanWGaYK8oPuo3Jv1E22
qQlPrbiSJEuIiEKkMXYHsVyoZyCJkRAHO4Kk0W/gplE4vhgleohFFpo9T/MJ6EXrWDfbFWKP5Emc
2xTBWop5awNM6FQsUjKZJ/HSQnFPiB2sDS0I1psWEppuBi2TFp28LowoE4JkWdtwiv7BLa9ttd+N
iSchy0LWp8CvvBhpFJVycbxLNJpHi/Y8vGYFQ0GaQFAqXoQykmPGiiw5z8DBg06rO6RxjEOC8fTJ
T6fQwxSwFGopFvR8Zzcwyp+frkS7TRxIE9j4eWbY84HmYRyLDT/PtOxZLmhrBhgIESY0cYKZnDuB
5sv5HDp2LptAaT9Xu7za33l/4g2Ytbv86ROHbBF+HATTJbZuGQQkHyCAIDkgvZSIIOqYp9l5Gma5
LB+QolFE8+rBeZyMypu/58Qg+mYekrDWN0leXqrm5e1iOce4QkjPlJ4+E1+BHNggCL6SMMCGJKdY
sxOSNFpxypNlNlb79bUcb01YKJL0laMkuRBjCbpbHDSSbHM800OVE7JUxJ/D8/NYKU1lS1I5WT8m
pSyZLGn7IrPg5QV//JIEGau7V7MIGycJ6jhPCDRvtuqtVmCSbMWbw/hqYoRaxnpBuMivlMCETLeF
GDQFEr/Q+sIpS0CIMBgCh5AzbpL7sL5mPmAvYG2U9+Eop7+uoZnniWgqHHPrEGloioCnqyUS7/tk
fwHGuSyAn+vh1cnh8Rf01ARs7v8OCe0SNi3h+L6jflO7dx8PTk9h0PTFwPnMij69fIeN8iKWdPnz
TMrf+OptUsyc4dMnHw/eHn4MvnxAmxsXWnLH2xf443bVJf7Q0y4ue+opXW7fPn1yeoIm3b1Op2Ms
X9xPonHhLgISY/1u7xXMkmXGjBzk/Vc+AGlh03dIGwIekBoQGv2zbClbFUlYQ9UWRP8D6CWNgSPN
rbYk9G29KbZPFrdyogCT8MSwTVMWwIGR3PoTkk/h8nwTJnHW35XtnRZQKsazII8AqrvXEtCrx7O8
v4NhkajDxg3ta5sECKDu9Dyz6RFpFUTnoOfsKyJZOGq0ggkEYt+xNih80+aPiJNda5fyHK9lIG4b
iI1hNaHy63a5X8WklrpqG3t3aAPcMQA11RR9NJzG1tZq7mYWnF0DR/PD3k4Jxex0lmAAt1ht90xb
i392iH80AGu/s2BAQ6xgHJfjKHmOtjunhFHb/CwoY6V5qr3OgleOR/Nsk8DNXc7eEEutooT3rDTn
sHeCY1goQdHMYUeQ+vWe7THSZPL7rTIWZmxjkgXgG2y7BttyHdXm8cWaC0G4u512D8tZ/PibeLuE
uZ9BiBYzizWOS+41i7EBUw0/jNtkyZH0NU4JC0bJr2YFN2CUngtchRd3eS4YIKQQiBiYNUaDbjGW
Le7nlgTpT6ew46wF2EH3dUFKwhYNDsDFWzk1VUooXZ3IfBkX/DBTl4F66dPuWg5K/zilTuuqryAp
73T+WczefQRK2/dgtH0nQnVF2eUvPUM62LvBwfHxx6N3B28/kvC+uWW+VA6HK6kdamiZRZLcg5m0
fG0l95LvxyUGLpmXBR9txO1/8Yc0HDgVoHSI0rUZ5MQP7Pa8dtn6CrAcSH7TY9ppvX3tLaQnPCx1
T64tnh0spwQuU3e9CSvqaEKf+bDR43AsXeeXgsgtHK961DJPGIa9m2PHh9rsFp74vi96ytkz6A59
WPEycz3q3HVYqBMIstscu/cS8UEx6AyrDv2r8JIaON4Q+NkQtbGV5zIruGtu7ol+X7zuQWJOnZvq
6a34Az9mj0xGki93tI4L8xIDD+ejCZSefdK13FwPzwkcD9jornK4ZfIGXeGTuqSh8fc+ZNQcKo41
LG7j57LAXIbgWhfduexikKCe50MKu5caPnlt4jClHlIGnhrgDEQDN4S+BKFF97Yx5RiAoTJ9qHrj
idfAWelC2zuRv63RFYq+2+WPrS/h7FtKjcol0Vv3yLTf3YWu5JRsq1G7yhIIQ+Pj4PUDXW1YERLT
zB2wbuyStujTr7p4sNjeec6c4Xk1amMIlvrIXcPuK4GQnGLNFU09mlmL+68Lq2WzNwb8AuomPnPq
TM+vyOliyCMwGro3sJTh46Jlk9+1L0hWT9eW6oZWqVmna0vQWoYpL8OdRlNDez9MKXTi3jhEw30i
PihKI8EN/cFdGWeZ4BnzEZ435Oz6D4DATYMW0zgJwe20egAMnVnPusOH4cThSMZoY9T0gUsLMx30
CJ663B563vBhhIhKNCwWLjSI4a2mGTRReI6ZIkpyk8AdsQYQGO3BDVtQGfu7HVLCZlGfFAW6hBCR
WX/HTA6s2v9ktWoGFhM7xazNH9j6hKV4uLUoSqhduz2vdAKSZkL/lMc7VorKPnmWwmUOh4bWXPJZ
NIX7C6sPjleOypHGBNNWOeHGyv7Jq9iC9eUVBzvwsfb2sc1OgTFlqpYeXr8coEaNTPcc4bGVDxfz
AshoG1oRrlXirj5frH4Fv8Jmegn3akc9m0V4hH7dWdTiD54j4vf69Ws9LSNgH+ALBdBlQrbEIIYJ
TF/TNKirISyTYpXKvkOkdeqzatBwGRz+Qdjm9LULry1z4nbPq6addK7AaFrSDa2JPa4rZjV17I6p
3PWg3MJFmJDfdwKHPjwFFFbxSzKSnFLrAehAW6Y/MHfRsVcfhxtisPNNyNNan9MmINuvlFwPq/EY
jTSARirdlLbMFnWYZEE6Lvq9XWZibAp9TIs12FOjyebLUcGOZoyrVGdNZJjhCVvp30yHPc+OmQnW
nRUW7DiBB3bOflUKCFWOvhG5Usj/TBEAvBqTVwrkJ3Q1e7Pvo9yp4KTOtRMlUjHdXLmRQni4yKfH
vpkqxECfhpNLAouvluzMWUy091wz/Kkev9RewVDtoTW6kDcaQ+GoBQ2K4yHsXKz5VPAMBAHTxDrg
kcLFSJ5yeGZCHm2hw3zpMjODGFHoSOYm+lZzJeENBwBtoGVECr6skfaFxwn84AUYCQ7pnG1X5Uq6
xGSSdw7vlmNywdVmyaK4IZYZKKnGEUZZJOfK404fsMYMfYa84yr8F0/bpOAp6WXc4gU4GNHLU4tp
FBvx5FgTRgRh9/7yfAZXFGK/nK4Av2QZmentmhiMhSzGjwlEAAPSpPdah0ZJji3elJyuPF7YZvG5
nkhmNER5DYOR+7iKuqF3pcfwBCP0h2B0mMVk9mPIbOy53Q74o0JEr6suAlJaBkzYg5iJ3e/KoFdU
KNpx8GGG+BAJc+zKE8X5avkwWyhBDG/qyoTLZ2woHy9zFtcUdSIq5iUfqMnl0B+inWK+zBGbiRVf
GbdhKN59/oztZKFCPtQrqUrghSqaBCRWGmoVfwYaFOeT2SUHLZvbg2LePhypfkWRNfkDcQytqA9N
8ULKdBLNlZVel3xKMEZzqKCKUdpqgp6bkWAjtiQ4mJ38LS4rM6yGwFO1mMBTMz23BNyZyvKo5yu0
1BpXLnF2dYdqfdAs5HiHmDceVOJb73kgCsKcOv6r3D3s+gMB0OvAUY+cYUvdVk4dZ2ja5GTos11B
Wg02yucEwDOvp1gn/LYL/bP6+uVLckGaTX1l6XnQVwLWEBkzGEvhRJMk65+ecOoNGLevCWQw6tMW
BGqRT0a9AXF3H1LT5skisSdOXo8lotuH/IcEJLhKWrjB92PkGeRWmwWlvYmwN/rKxOO0m1CH820o
iKJmZqXAUxQr0e6LDwhS2pISSxxbTLEs2PdetXc/UVJTcKZdjMFxlhTJOIlrKTs+62Ol2hwSX58s
F7T/HGZZkrkOxURZGAqisgqJwghiDUkrXFodD9k06tRIcThPC8rGGHNENedIKMMj35cSnWXuBVNK
DfJelKaOZKiMioY9UQjdgAluHc8IeiAGliTfu+sYXbdmkIelBmbpwnZzPa4/VVy5XzOMmOO1m9EZ
spnIvu/9O5jpmYoKlzE24bJnUNnNDSsnVBKGZQsYnqI8/lhGsWutEI0g7Lz98umwHn3YF9/082wt
RrgRK3xESFH4oEKnXaKjlwp1X6d2WEd0fZ6043BtmtZ0102Nteey1viZ+IFydrSblYxKbJ6nZx/O
4NOaY2206PkCojL5O1YFbnhP1tuqsULAHRdmY64tUktRmrHBmZssuyqSa2ltV6GtEYaFxekkIxRt
SUU2oi0vpgURGkJyWvS7nR787LMkDUDo82KGSEWHxPrC3O90OghMPX8uehXguSUppzJkVYgkdiWT
3NN+XZWmO6+UptrFr0X/A7Zo/cdGewp9DjYmXcDBTjbmZjZ5BL6rfngHdnaP34Tp/UStsP8moOVI
W0oNZDtEQYjtYfLLoIAjZuTOKa9h2leKgWfz+MBRwQlnaFGL4bj4jcU398ngBAO8pCdscXnWc2yu
ZEZtprrCZ46mr7Ar0qf4/ZCoIMVGTN6WaiwWGYf/tZ7NEP0cC0MOzEPuRwuQGGN1VVzSiBKtCrTF
macirGcsfPVjNgDj+WC/JfbNw2FNn8LLz9Ckh/fawRzAD8ZIyZKuyrkDdSEHwlKHeqaTrCwV1aTZ
qqSpa4pwQeksaBPXsuVCrihtjuHCfi94UycQMb8zkNXeX6WMLOQVZ+BoQZHgJW63cltryK1kP1ar
lVZAMQItQtABMQIcwsQpW2o5bA1v54F6UKlleJjrh3rnwpPghjQzfqiYbGvo3To1Zp86wSg1n7G8
3TKb5lZLcMDM826DyaL+jZbJ1idrYMNGE70H2E0264qlNrmmeXo17ZF8lE3Ppw6x82w57EnHXIMO
IOStv0hXpks0mocXSJ7L8rXkALoB67L3M0gubDWRuZ6XJrZEYjJ81201NWYznmqdNzyq3IO1TC2z
QSI9JeUPoOKyi8gwO8WvKYet72QYGWPS59+6mzQjwk0dlSuKtUAmlmLbmzU3MmNwK3zKQtBuHb2E
H4XB1YuNKJATGI4xNowlrHk2uxSd7K1gPogoJGLMnmzAzlwQDlfKC6uuyfeqDJA1J1J3r+7LdiMS
bp74TnR3EQ9tqKxN8ghxE73o3r68KafxtiIDOoB96q5bdL9vav6P4mpz5B4g6U+JOnOwwB3PDWHH
evrCPMyycIU3LeWlM5xLzvfxfEByeYj9gC6HPgVJWCDbtyyPlYfPWYQLQ5lwtFAguvv6S5PiTwuC
nmMtDDENuETiSU9fIrtimxYut0dP9OeOHoz/k18gQCkoPE1Ne54eDB5Wl6ntBNU+/3HkIm8IFjLs
ELj+aS0iNanQ27VJ89i3HcIX7EbwNYGNfUxp7vA+QO5Aq8uia9Vi+aui8nIRweNHITHvjcARAgp8
7Qtmh7BYJAuyelyOz0HHVSGrlBbF8lcdpMqwNhmSShHxtbEQ4DlFZgnsZSPOFBCAjLYrVw2qFj9S
0KoUXukOoutBOqz6BmhfJdK6S7iiifWXv3qkl3DAUom5obWsaFgVpzXo4a4G+VBRmi846pH3ib/A
Bz2e9HqcCnN/TfS4bizMSxPTua6zwSAj16arJvs3ILqjCPkbDabupIGPYNDzd1vi9R/93aE3/D9b
dZRhWCYrEp/hQYAbEhoBsZdSNJDnV0o4k6aXZGPICf7jL9ifsFjoe3JimlRQlQbPX8GsiHOfAugG
yHtcw+XR4ouPsNRl9u2ci/XSDdjvAETHY2yn45W6N8YWBKxJBKOkMPAebdrLlLIV3ZuLfbAiEf5C
B10peWOgCDC0Y7tk91/4LMDJ8TTjILSJWFGuCDqx2w4cziAxDiUEG/CBM15OQo4AK6LQrR/lQXgJ
FwN5soyAcsbp0vnX9Ae9I/yyuNnqbz3/487tLwvsDAq7231xQ9jdVm8rF0RBbrJBxlThzY7DwsCZ
NjCO2g2t4LIeXyEf2YgSDHSbvxbwPf6VW9rqLT1l5daEr1v117J8zckKCtaKsaYtgjeIAbpVocph
hVKRDTH7q0I+/KU0okKlIdS+tiKwa03MZKtAT4AEPiz8/Xog2RJ1vWYQear9jYo9SHzSHcROb5ez
HZGaQDzpeuvh5Q6sjin+XZfWy4Yo87RzVwdVM1gyU9I3Gh2+EddkxEw7+9POiymsGdJyOo8fWnFX
z6+6jxlaYQ2tt2loxYND69HQiruHhv+Kzn7ReVE0hqZl9nUpUjj9/v2pq0WYPVSa+yCgpMggcCE9
/9oS2LPBCEi+yn3c5T7l/vp4gk7Kt00A2MCoPRoZK5FyZvyV1/zwnM7tyLnqLGoSnXY5RQ8SqgHn
U7uWIgWMoK62hGVzNshvwp0VP/tjBMmkq2xdNRCWWdee2RGwRyNhxFUpOyt04FVLIw54pVYy3wUd
WRKs9MIngcG5rw7lFFg5rvyuunfuSTLIZ8vpNDb5gBh2QAnYOH3VN84MQkRuQEQSItKYcQ/0vwZZ
q9l0OKpfJ8bA33mFNezv7O7R787e0PMvI3lFKczbrFZ0jcpZTNYb93qv0axHTin83t3Y2GZOZPi5
i4X/iRPV7+bPtVSvZUo5XH75RYMbrmgTO/qE0xGfD8+6fw7+0nUqn3KZQQy5yBxBbo0GfOMey80Q
lbrg83FmmK2cnriQBRIDXZUbnPevvLLVmmjgnlVisLNR4BkFseoZxiJtqHCR1nPCavaUr48M5wF8
GTQfzA/NwZxzWhjofDAJUwq7HVyeHydJ3Ju4KjUdogVKj0rGVl++Vzeu2r7VjTP0mpCtNGZu9pGd
wC6ykjukKxJYPtjSX6d9fa5BgCtEIkk6XG+SDteU0wfKrPOhnUzFnW2gFUFwr+m0DVjeLxJIB5VW
wR49cHLt2UbJYo/VLYnlMnHdatawj3psgGBJuF2vkihqoczDxRKuYLIuFGnpyqH9vjJD1l7q1con
6fu8ZDzCF9g2nJj1VPYao2kHuImI4hJxcpxig+xjgQbXlPHJlna+Ml3rlKaVNdY2KnyhO6ThAXt6
fIUHVz5R2F6O4yzSLPWOjnQeIi02SVcfcakXTr8mQq6MWa+3AyXtWzoLpk9j3uxlvacf425Ki1Jg
4Tqa03KYu4O0Wn1M49rC4zTI+ip7IGkMxx54MuKMJK8aI3Tscbjqw0e8Y6TneNZAJs4COnRIgjDz
3yEdYyEPFlhNlE7/8cTFRy1xFpDbm8GrUxQld/CZsKwECT+f/wNwPeXHrlLoKaBN2vukf6daX7Lr
iBMw6HegDHm+lGlLKTDtLiUAfeajDLigEKJPv1zLCSXTSs0y/jmDNTuKbHcUE55lg808BAa2E7u/
eUtuSgZ6R8JBLwfc1ldGybYUdatIEy4RsAWXrxFGkZGIop+43ibZy6dA+sxzLqPuklaxanSKKfPJ
CcFcA49CQXEIOCa0mf9GT5rPf1yC6fmUZcOCsClq1afLBV8ExA4kNBbaYMXRi5S74UBe4K4zckvs
lhGSBlAEvFMN0HSjzE7rc7Cm+s5rTppEPq+Lpmlm+UoaNF9oGqzZEWp+AzO/cgOt08z4JjSlzQxD
TmXnFInpej7sUAhFpTfaqmGarftk0qw2rvsdLKRjpdkjXSsA9j0vlMYw7lpM6DofK/gyrUHCu++o
7Af5UIVaS/1+fQnd41rF1zcy3e/0JnCvcuRBNUI0QlTFU27y8b6/M8Ujl0u13DDm9MSrrGx2aW4m
izWeTdQxcok8fi3l+rRdplq9TcZ8BAVcGiK0jwtOreCs4amD2iaBcQXw0Y3kCs+N28cxdVoCFUCi
NuQtuEM8l8dMAquP8tDKHW1oGaA+DAXVC9WBPjzqlxc4yAHWQy7clAWg893f2t/N299NnLvSXxwS
uepsCYOEGzF2mwbFHU3pmGfOeFPZF1NwBxTZWLlmUpbGaFSmcW4bJ15UQRGagTVPE7mhTHJ/o9VE
2ZVQwqcJz55+QAhyaYGg1+n+0XkgsqscL4hC4LQwz8m3lOx5EDjvKSYvAtD5lEnmab9M84X0HoTH
XK9Qoma06qhp3SHtrdFKL3ksQ6aUuSq3VejGyF3IVoHyJBIdovFoFikvYWBocv94G0ACPmrBS4a8
uLQWdzYjpt2XjNrGbuswHiLSJhAIC7yms4B2lKAhRohyjwS9CTX5DailG9qnj2lvvLdlM+Ubrzt1
GyPzvMdAhod4TnVPGpCN43iNWsj6zBBD7zuq1bflT6z/sKYyiZShCy/Bo3Bu7gxBFpKIHs+xSVMF
i3WZIZnslKjNzFbgxH4csKJiVhMZFynt5djsvft0c+9+9NTS5w3i98H/vQRdNxkehSmlEtIWEiia
5CUfWNo1eQb5XPn98M7TZUAhZjohaen6cMIFypDit27n8UN8ZCxgbbrNHsWTbW5Ak2RJqXzO2FST
Q6bCJKgl1ZY1I+7j7ApklsQ0XKUULPjo4e/Y8BnmHXrF7a3JMzmlhGvONKRKLSlyHgqua7SV1woa
6TMwOedJcyrciJLX2iqgxQW46oXangmNPVcEKgs0cGGGWUInSKgrk5qOSiTmGDYUAOCYoCvOGDfF
RtRZbAOb6jzojHJ9CiPnUkpt5GXweQBdIOffxc/Q3MkTxB0r75bO1OESGzQMVD0rD/coxshBFLfa
2Gil6c2NNlu+ZDlWnuSiR5v4WOuAlNJPH6t1yXcBh8cIHJQSEjMcX7G1wtt1eBtSU6YKHVu99FPU
dTCWAx0K9icwJlzopy11lG4jmPIssQ1JHylGvqrDh81IYen3TJpNqgi1ERzJ9QYwTpdpDOrGWcEU
5o2d9wTcUkujTajdYe0E3m2LsqYBNo3oWJedUWPbDu3vN1gIN6DDYMvWFLaGg607d3m2M2q5SFPn
FBkwcNgwuFPYJ3lqjI8bS761EdnwO9Pb3GvkIwAB63SSqR9TGpQbjoRmZCOYTBEsXz7Hqg62bzhp
+3B+EJsr6IOZg/PimS9UB9jPElUsYVlM2yjJAIN4RKbTZFBXsIYN56J1nNJljNmCQdvBZj3sXufT
aK6Sde9Wv7wa6MfDHWzQm4YKVqm2DD3L+TN9+DiuThOz19BzvXhqTpcNRJ/+PorXqI3v2OpECJwk
CEZjPVnbAu6kzz0zdec7NRePAXknvMfMAw+X5sCFhO/rM+o4B5pVh9JpiyQPCR1zRg5Yb7gpMt5R
MXHUgPCUz6/0DijBgeC9Q8eb+w68wd0y+Vi/FMKUjkW9hIOzIxQHPOPKCW5Z9+GgU9Ywa1UlCyo5
pA7hO3W4G3sjGXazRUme+zs55NZW0yLGc/XCAN/a//4VP+BSjPquIbtutk4l3uzxd6dpefmhqy6d
5oiZHG1C8PWe1xBJaoHTv3FkfCOUHmJobnM++fvRfT5uE632X5DE5EgWz5su1/eZK75SEiuSAP6g
JkqrZKC6NRTIzAsKjOFQ6GTiMRxqEnHVSVUORDVzbAdaSVZ8q4hKIxiAjkNNS+wSr/Q2EWtqAtk9
f5seQc7rq2lXXd0QHt9Gs6sZu4DrpTc2SPEN2HJ5yqpcB/HhDcDd3s9NlQZwU6qRpD2CU02REtIw
zenwf9VlQRKI9P3BjdFmsOg2e7gypD60xJ1GOpbwfZLc0UZ4huPvhD87BwY7fJ3y9S5d26boYO9+
iLxugss8UDVilMlCeDDPcsaYYVhmUhInNS5liXJ/F6ZwD3XggHmdUjCtM6/ScB1TiWLj4QFbxg03
T0dQdhqQ040mWVV4oQoutROWXMOLlW9ajXQYUxeKFgkfcnUeaWRRr7o87Sn5fHVOIJempZqVZbnZ
nM94Ef6PBr1eAyn3N/EhRhBgg0QhJ2I7DLq+5NbV67t0Y/bdlsYZn4e8XzGuEsEElfsouLL1ZgiW
IoizwuSQc+3z9pkxcAquH02HAHRR5kmiigmq0r0LTldDYW11fJMNnmQ55lOx7L0sjzUfNMpTW5Ov
TtjR4VrNADNVLZQOIqA26JVYpupIgSoUDNPtnx1eByNp9FtV0I++cqpCvRU/yZgKvtWNQSqHwqVi
Y3a+KDcvlQ9sJ2nzSG1CMp+UfW2smTwxjsuVaXqI53Es/PTEDBoFE/hLjNjt0alZZHtG+Eup3c9F
sbGkgD68gjOUqDmjisCXZy+5bMMGeFQK716AKFW5LBigqcLMh2OsAggI4LvNw4ZxwuPRb+lOt43i
+KG2s8huS3dNHaMqxoGfXVX864IOr94Qbvu84eHcLIz1fyIh/h9iT1xUXwADZfvoLzqlaoO5+re+
Gt73VNSCOYUx/hOfqDVzA9eSLPgIRMLVLzrdzeUp1BH38OKOahGNk38M1fPWR1sWrlBaxLX4R/gP
TrfGc3skxET1oZhe4LeilxgGYuDbZhg4C2Sfzid2z39dokzApP0DKpXkcIXsl8UN+DhnlFfuFayl
qgKAAWgVAuCTiKNo0apKh1en/emWDuTR6R6SsEJt1Hyp1xr16j+cKW5UBnN0nVK8raIDLkfKXari
1+UjhBsOe1lIWzUuLLCU9reL3e4FivJQAZnHHTclvbAs2SI4PVfBPOczkv1GlQ/uyWhefNQ+ltOi
ZB1uxPmHwESnedAiMfhZj71GuZg7AG2Gswam4kODr5o5XQlELaoKYXvx/Qn/Kwhafg19viKKalxH
1AbwPRZhg6Ut0tCiVJXdG0PlxWu4nDmsr0p2WXqtKml2oKru1SvwtbgMo63Y/q7s8Mcnhhs0uaDY
/+dJt00H3gbVYbdh7ZTboDzhRs9D67k5xkbw19lEnbdissMQoOAcXeOkFYkPZD0Vprz8Qe/lcffl
MX5vv4Q3VU35bmO+y/ZcnmzN8kWpl8OPH84OT8942lBIlpVTtFXq6IeDo48NX1bH/qBbaTNzzn7R
s84pe6Z2sX+QnXMp9WO6q6rXpTRhQahfug5KK2uNyKFzBlyaAae5yEhgh+Hd7cjsaOka73mfQzfE
RN7dLeBM/NZOSr/dtzYsa6k5LXOkvw+lUy4uI6hC7PSq2EyHng9+en/0JXh/dII2mfNu/5efQLn8
l7egIKpS//Je5hdFkv6iPkZ5ZohBCl+ToeLcM+yq+uFdMQQLwXtKU6p+YdfFlHdHFKA4NU2dqhJn
MMCpFmIEOtaLqSdUODJVJv7RA9/M+vpxxkpDbrQwU7GhSeVdrTlX76glSdCsWncKelUi0oQIJqRO
uvbRC7bZtGRq+F4f4Ww3fnZvuJYGqlCC0sQLbMC3GMNwTVAQYUw93JuqVmHD/bXFo9ka9vtb1GAL
guQf6jVKdyCWg8gWDa/0KFRnLfDUJq85tlQ7saTTZGyac2VxSn8mCwXlzcn5FgQkHVANWwOc60y5
/wW784oSnGkAAA==''')

import base64, gzip
_src = gzip.decompress(base64.b64decode(M45_ABLATION_B64))
with open('/kaggle/working/m45_ablation.py', 'wb') as fh:
    fh.write(_src)
print('wrote m45_ablation.py', len(_src), 'bytes')


In [ ]:
M49_XVAL_B64 = (
'''...''')

import base64, gzip
_src = gzip.decompress(base64.b64decode(M49_XVAL_B64))
with open('/kaggle/working/m49_xval.py', 'wb') as fh:
    fh.write(_src)
print('wrote m49_xval.py', len(_src), 'bytes')


In [ ]:
M50_OPENSET_B64 = (
'''H4sIAAAAAAAC/7Vce3PbRpL/X59iFtkrAQkJS7Ls2EqYLSVRYu/5kbKUZKsYHQSSQxIRCTB4SKZ1
2s9+v+6eAQYgKXt3cypbIoGZRk9Pv7sHn/3lUVXkj0ZJ+kinN2q1LudZ+njP87zXTw5UX32X5N9l
ufo+mSXn42ylVVyoWE3jvP/27fcq1bO4TG60GmdpmWcLNcXYcq7VSBelevndty9eqmU20Ytwb+/X
F6cX6uLFy3OFf2/eXuwp/LwsVVLQNwAt8zgtphoAMDm0T07oeSsglY3jfJJkszxe4nH5qipOVFLy
k+MkLdRcx3mpiqxKJ0WPgcfpBCMKtYhHegEwuVbLKsc/tcp1odNSPVLxyHyo0us0u01DdarGeTy+
XuhHt3OtP2Bti7gokmkCzOZxwZA/6DwTsCq70fkiXqnbpJxj6XHZAw6qAIZJOiMEzQ1tyaHLPBmr
26xaTIBHNqnGWqUZg53pVOfxIvkAomapSqvlCA+NSxUvFqG6en38PBrnWVFEk7iMC10+uiLyAE2s
rKbeAvtRgN4E8E1WzgkNHpHreELENIRl9EMawtgZysRpmpX8+B6wkluFni1BJEEqLIsbIa+9PdFL
2pXVPBkX9bZhvM4L4pYsBXKMRZYu1icqrrCLTClCfqKnSZoQZKxxzTBjho4dWiV5XGb5Wo3X44UO
1YWZc0t/l/G1Lvj5+v1K5wlhqG5AvInCrDmx0TxOsdxxXOoZQdF5nuUuZwistboaJzn2K0rSiX5/
RZsMPMz2xvmsYtBlhu3BbuZZOrPc/PJCvT47Pf/53dk5g724zdQfFbgXyxHc6l2RvSyATcprTItb
nZ/ILh2G6t3Z38++u3j59g0ICKxp6jTPPuhUxAcisVgIzUDKghAnKMTck2wJ9v8biKPVKivK/jwb
K+Kj2ZqB44eYUaur/iKbFVjM+5WPT5gbXCn/VVIpTQwW9mSv46WWCTlt3tUpOH+/ePT66PlVQI8d
Z8tVVeqJhU3s73A3JuZ6jPv9YrUAhUmUZfsK5bOABUx8nmZY5RZ0z25x34hg0LPAaSRTEXSdJAUE
Z1QJbUmWC72KsdV6okZrdfrzu7ffiaxBXYBVsWf9hb4B7UZZVmJuvDJCcQQZ//b84uyN0PvXOahs
mIE2tweSAltcIWz+pl5rYiNgstSTBB+X8ft+kU1L/O0Rm+fZat2jARZrwhgKZpIwHVh7KGxaKaoL
ItTSVCqekfoqG+r/UcVpCZHQJDtC19DChgqAjq0MXYQ3GGvGELpwmkyIWzHx/Kd35zzSdzBWB+Hz
48dPWKeMxxUUXc0lB+HR8ZfPAxIz4TCRrTLXS+KshKBOhQcx8UaDmhcvztR3p7+csWLHr9c/n+PT
O1x5pX59efFC0YA3P7/+9uwdP6VR6T+cvmMD0sduTZIpJISxzmdxGgorxxDq/QIIgHKsPFY67RfE
qvmSgflX0BzxQkY+IiEISxLfAvxHigr/gL1OC2KYnmGQg/Dp8dOngaoKUPfNmcHCMF5xwoDnejHp
Z1WpFhWeC77T0LViBTQAHT63/AVt907/To8j+su2jrToKDAKppHNgKa+Zl1k1Fo9IcVzCw3m4+cA
5jyDXK7ZftRaWtBmjiQrmevVIh7rnqpW4KaJBskYbFGBCIWeaN41WWVorGtMJMnSSc9iZDYzNbRP
xtWiXKsYlO5ZzYilpCQDYu74mSRlkOZ4tEgKVudGHZEBKLEmIDut0rGRT7ZZ2N5bWIW5Sgeg2phN
HumtYp7dCkt/CwNV60bSr1hglkNuerXdTHK7P1gHFF4KGZnAGhJD4ELYLCECbA2TKJQmzGBEcKNQ
fz9/+waqN88TthkgCll9nWJVIDZ5EaKWR1gM5HkFnZKkorzHebKiNYHdz09f//Tq5Zsf1bvTi7MO
Q0PrZfkE87DyY3X94kNPIGPb1n9U4GEadNQ7ODhQLz6ALvp9PC5h8GoxBq8vIcxXU0jplRCH5KDM
ViJ2oD/tDa0B9h2PAfqV1X6yVsaAhIb0Rx+mENtN+qzWLKQfSCWIDVyQQ7BuMCff7cUP0StwY/QL
bNIP+B5bhmVpqu0sk3BclYWBPdXqNl6TP8agG9ssxn6RZdfq9dt3Z3goBMdw+DyZzQ1/Axp0OdGD
FnmbTEiNk8qtCt4UwlQZ34vkBjuUQ69jKZbJb/OEKEJm+sp4R7Dn0yxM2UGNsP7VQkdkL4po/gEU
3vv5/PRH2UbxetXyyUFkuWq1Vn2om8WULNiDg4zzkMPKqDAMH8l33EjGo3kSCQXoBn+KsKyofF9G
0wQ2kfzsPRj6pYqiaVVWuY4ilSxJBBxPrNjbs9fyGehUaPsdK9UltsR+ny2ykf38e5Gl9nNW2E8Q
5RIbvbTfi3V9iwHZL5BJrI48odXe3oszbN1A+VkRkpqCyOUkhfV3ONH01494VVEUwFWYKs9+9UiY
CLV4UfiBtTcKPjn0VxFC7Me3Ez8I9vY+U0v4uO/hxjnSqsE/UHW5MDKQv0UgAH4pitpBEOcDsYj+
ChCeRFBS4q06UAi4JnMBK6zFOTWza6dFsdMClmIakI4bkX4ytlDUZr/Wp5gHEolwAnixAozcxifQ
xOLikywo+CFwQI0/z854DGTioiLjag2rxgJW4PbylowC+TyifODMk2hGY9bNqfJpO3qqS3z7/Xeg
aYZ4Yejh90bc4AVBILYO22TnsaUt2mD4mYBgdwUcj7lMOEGHvdCUuIhnndSba6+E8G10XvoHPZkR
WAarNxo89g+15eczAP8jPlFnxwdHe6+Pn4AB/xHS34d+WpP2vnt1en5+ds4zzee9X1+++f7trxFd
fBYePATJVc1JuiKHQKezcv6VVfs5ea3kiRoPdo8YePAf/yiOQ/YQGSmjW4y/4RelXpqNg+IwWJAk
ForVibr62gyNksk30ddxVZBhZFGIFgig6cM34W18A89fh7NQHT15fBCd/hIY1/htqq1zIyE9OdyA
W61IsU4p0rdgCtbiLJ1QziQriJDB70ZYKNazgsWgay8c1pxk7J1mlUwTxQLhE9EZa4SuF9mB7wlz
yo+9STSIDTmhcHI8TxaTfTEH4zmFGHEhw4h0UM7iAyO0z28EWYhtbQVhFox2KB1Hk7RzaZ1HkQ1c
hI0pQ0tw48BBTYPjsRUhqwvfi7xgeHCJ3Xe2jPfQJ6PQM+wRFQPLez0KIiLDNYM3WBKuJGmERMYC
ww5DSAs0xigr9OAir3Sz5y8JbO12pNgUSES/mMfkshA/UkDvJjzIcvagVMbgBcozgHp0GV8mZs/P
Yjho9RYoMewCGywAglSc4bmyq7iy7E6pEmxowns4zWNOEpBvB/8tNy4YjPVVszJOV1gHy43SS3Dv
xHAZsksVXEd5yAndEvf6IHysCrMF5O3DIpC3CA7hYO2fR1+yBRO1vozTagoPCwqW80dwXDIEZOLS
EXv3jGeaFNYVq51GOCKa5oqX43g9hikpXoRPIvqbSWaY1mhxtvNtpoHEFVA4Bbu2PpnCkH61la0w
i/f556SzPycp9YIe7UyVF9gB4YTAqm3SvAS2UbnYCmzxDyAlcjo/kHN0RhkPf89Va1MPqQ2CDTEk
Ct7RU/+S32Mzy5I4gVbw3/FsttDKWAvltSB4+6PfM52nvyPZoJE2xATjA/UnyBIWlCXsdxJ2fQOq
f3O0D8uVI6C5Tcmqd2HPy3JVnDx6tJqvKW6Dj4WY8BGpImyAca36HGj1OX5+BFkJHz/yAuFlkoCe
YievVzPaQA0ve+ruHv895s6o5nbvREHUPOZOy8F07V48Pp3TUAbAF8gME+lg8dqkJ22AYXY7WS8g
+Kz3d4TFs8eEaQErCztzUsGvsT5qzpYqKqbsuvJgZ3exqGE98hJD+RK5T359uacOAvWFOqyngVPw
CPW1o19OWiQ3VBpukOZSfTFwALGuxT4kaaXriymwOKi/lfQtbL5DwsBFJR4OFNpPhYrGYODkl0DX
qpYeDQxaA4E/je0zmF1raK2jvZlbVsHWCHrxunWVlWe8IuPh33kkfCe0yeAO2lx8oT/41hhYXNtm
n3t7O30KgALnYh7JOR6ET5r9q3r/cKX+fN+mRLq5EhDHMSSs+FL1zcC9ePIJSy8Jst2D+o7hfkuT
1IjYivTtAHpj6JLikmWDTBITUuQHq5WxXmPhI8p4YJVwpYi/C2g49y7ZJXPXPL9LT88szIyipwXN
ttjLhOWumZGBLM+MJK2HeVMoJIjSKpQrNQIczFhdwEELmHw77EgiA8Ki4WlvewhKLCX2gK9sYGsY
GsPMp+4AsfIRWXgM8ti29zfz71XKA8nAwpnpZ9O+pIzFTnn31qAYh6NhmBWMW+lPvTrNcVcT/L5O
2nLweufu173jz/U6+p3sz129P/d1Is0LtjxVGVFQQjhRdyfqjhKrvku6EMp2icgyCO53ALLYrthn
sK6OLxsdnGzDkpl3uP8Qz+xfnoSH013PNLtGCMsnO9B4kGKr+DF/VuDAefuC3VCpANhEf+0/btYA
eia2bVUOFGVFpWzgVAhslPCS3OtkjNgNXETJBM668PBI5lMSzq0d0H+bNYnIGMKno31brdPRlQ0k
xMTayqIbK5s4PdXtm4njwbNX2xQwKOZB3nisxMsXdUR3TFVlhArfCAITqhcvf3xBvhxihCWt18qI
U25o+3LM8cUYuIcU8idEBglpa9q62+xQHKolLpCQiO2+wN6V65Ue4AYrn6fH2A7Kww4OAxNOkCo3
ufuNzXx9/pOJa3A3WVZAy2T5UVccxSN4zuV6c4V1lYAJz5eS1CiF0K7yA/T2p+DLg7UMpjVSbvND
SNge9tS11qtJsixc19WQxdeoueoQlNkyUOZbEpgaS3v5/2/4rTD2AdxaW+uv1OcEGE/HR7hdun94
BPR5pkUfGcA8G/siHpzH7oloRbbeVW+oJEPJkpraFmkpm/z2pSB5GAgHti7C6UMWWETwKmxzKirZ
SOaHUncuLK8CJSA2jngKj14LkfJoiN9U4C580tQO3gG4E/cgNu4tu4pAnNrCBWO3xwXSUxs3LIhL
a4sIOkYhU4Byqr+GCf5aHTmRjtCfAmd3P8SAt1bmUzklsFvBOxHN8gzVkkk0TtqbwteLrVtU37Tf
t/t4aURZjsERsvwAoPVkcHzU3d2HS5QntUk00bvYvokU423gvdfUTgpIcmesmwRZrKnqg7COC65U
jiCusSkeymG7iRibx2KT6WRgJH35rG/ynsUiQeGknYtxzGrM1a8dmZi6/C7wpdwj40saswBSN9QE
YYL5MsvgBiCOvW1r4eIae3LdVgI7uczd2cDUyzC92ja9LkJvAWDviaLgdPCglm8gVFRyq8LnykA3
PDy7Foj2qxmZ0BLuZieKVVZcwokj0fNpaQM1C1gZUIYBMI2nVu2YUW3MqGRGjj0RmQTds2UISYiR
FYxw3ScmFUxu2mFuxK58nM60L1wdNOIXCzjwAvVWwEnT/jC5Hs4um2cDdjieZ+ATn4hB8lxdczaD
a4miSy8bx2m0DWS1E2RlQFYPgHzv7s0wvqTtGY6cAdAz720nxRtOjFH8tAqTgltStP8+aEdPNzYW
ei9AxgnRLKc8hC+65wPwOZYt+EA4Axp8kzG5SwvtI5gcHoVPeur5l+GTy+CSvW6JKIaSBKTfly0z
KdCZ1xg0hSLMeC7iDII+4fHjxCYijZkf62iEpO31hvvwq63fShsBCtywcpLyNHU7pNW4GYYkkvRZ
4x4sixU3P6jBNg+l17XZIjA5J2M2rHYQoqxlLL6z9Lua9h45L5HzHIQ7LtWBTEhD/IDJ33Mnkp/+
wNQm2gOQjenIInA6GPUk6M8IvRMHnfk0TX1DXRUHwQ4cNoE83wnkebBzIViGIWpnNq7umoRlmzkR
HPwiGS305vrJeSFhMgWSYANM3csScQ9EVPeyANadmTVMLk9Ihfu+7DKKGOIGBcGe+qQfkpmkUTot
jO4fwMaSt4tMi8AOUg6p9v4sxO7/rABuQsX9nMU3r1LfKS1DsK9X+A3hjFB+7SmnumwqCHKFc4/m
yiJZ2s+Npo3L8Twqkg968PS413ZZPqlQsaUwgTQnlTmBF+qHFkGuJ0bZteM3s6LBSqZoSoA/GnO6
k9LAEao4WB8pNp/WaRWGjaab2+qOJg73Ka26f3mPOGKVwb0xV/kLLrfzDlPPdpWYQrEZba9GhnJk
/ymmP57eBzajbDLtvlvM5zpsQ+ygm4L/JV5Uekvu3XsAiHHh0CqSU3cLtX6YnpdkoqWyYLrv4jwp
qC/svJs7bwrh3HVHDRXcWEldOHA6Sa0nrPAp4OMWRzfIpRh+xOEvnhba5X/GmFBhnR5YtrBCD46o
6ggNQEi6FtQ6ieQXB9cmMi+4NmNbTZYo41PQudcUWMFN1Fo72V7XJ+M2hYsi7UbadIsQqSjBnnPL
aB30U6+XaYf6rF1dQikd4lSY5jV2PLH0eJZrypjZhx31nj5+apsV6RkTTXV8pM1gQ8X7ZEIQ0zLa
6whanffQ73J2Sz5botl7WOuU2WIgSgtChY7sBM19nOKnfuzoRzQeRRdvX1GFCEnIgyeoxn8EYHuL
WBh7rhJoPjaibf4aF5Vxz6GSBrx+VApIUDzxU5gdIsllLTJxseuBdNmD4yXfxNp7TZBH2qmRnR2Q
msvDE55x2dy2VzraomHRE0lNNjCQc3RExDamcqBcGkbftzFSUQQmYzersSHnhV2XISTH7+TAgyYJ
3jzx0kiSiR3MArk8HNkEeatm3FL6tUau08mfaE7Nj6u+nc+7tnpjU1ysgaj7dQfx6169hvzuLGxA
u+3XkBhbameSfBkrb+WqTTyWsu0s/BB1ukvucOTMQeBTt9nFy260SXNSlHZfh1NU/OupKRIJsI2m
n45aeXxPEqZez2ReIdjUF6epFMipOeQ7veDju+57cCK9nuuNA9Qiu/03IFk/02lv3oaUY/ck+MXd
aepb4Qx65lsrahbtLAkF02YOW4TDElSwmEgmGErayVnaXRnpaWZyvHXTYAmvI23VOi11Qy61FWT4
fCGD14nmapz79KlfVE18W417EtxtJowo7VCniCiJ0KSIjCtlwuUamuGHIbHAJVfDBKrlnRuTmPBO
5MHm9jh5/oQrjA/VE5uhEXXVUBnIcCpZCKFqbLzjmjQPAeQ4w6KzESjBcy6uu/HBDiBmebvAVALm
fkvp5I4odfKsuDf7fAey3Ku7cXLvtUVs6HV6fz2ib+NzeRv95if/Vru5yVIVjsPlue3kVGTY6CWH
Y/GvNJI7oP+VnvJP7Sd3wLMjaSoo3FdO/u2WvvJWT3lM/eJ6Z0e5A/4Tmsup/bt4uHO85zaOO9Dr
HvLQ2I2aFag4FGXZhDwV2t6xcIORNmrq4L2CWFjBc0rGh88/0a566DpBPocEraluIfPvHEXoOccO
bD+Z96ngLW7UpIln1BwkTkuHhVC7ZbCTjHpUWvmSMqZ4uB0fsC2VvgV2JkU6InuGLzJn+Dq4mmkk
k1Qj93Ye/UOM0onNwATd44Nd4HWTYUatD97bX19fvIJvhtxCZyB1OkcU/izAMlwUN63PIX0Iy2wC
ax0gd5dRQRKapqOhvKSAM849ITz9B/Qid9wFz9yXdkksZZ41hXUfHTnCVtzaGHTxI/VDnORvbLS3
5XQjS4Jc33rmj7v43HN6JHXeJmgKq76yPcw4Ebm97C8BJzu5UBKs9dzjQ/FW0JL0o/6+lEMgKhSI
FmxOBPi1lmR1KI3B6EemZi26vwWs36QlAy5q8FESSmCKInYiffLAdbe/G5KWF+XJFtAc5ct5yom6
q6OW+/qAFbQW2IxufjT8D11Hyc0+MfYztqbwTtkRbXd+1I66m0EzcZzpLuuIpe3Ixx37MbQffG5t
4YMHEbajEEveDGvd2WB53M5yeMbNzCiS8l8U+R7f84IwsvejqDN/kYxQBYx3zDd3d0FwqeYeyNgk
gG1KP7EGO9qpLqBnWPdOSdcj0ZRNIzeH8VFdRk4STYqhrdFgWLBGu7NbdjK7L8x2didy6N7SCXJW
gE7ysC8b3yC8xrED3W6lrZ3XNTfTlnm1zR54RulDLgi4cPvRweGXztGEjGwpNRs8PXh0fNBJ8XTh
0V3Tb2Q7olpB2LbhUuPyJI3bLsDOmiLv7qnRdcIdbY0HunWNLaR2oyR3HsRpB0Y7u62cmHu4a9Rl
l+FsUTUqM4GCfuvw/F1nWH1qSdzQrWagcUb/k/NhdDws3KZYP+2w2MdPdW0BvfWg1/ZDXhwdNoe8
WsFavNUgOGe52ge3U+lQpv3iQzKht2HSmw1tb+92LSSuW8TUJC3kNbYGX9kmEO/OKWPlcSa5vizf
Lh8KerZZk3r+1psPgnMGRo1diyh5S2utrZuzPuMK47b51LVbbHVl5ZxjovBys0gYbRzD3oEcif4O
EAKgro6wo16VToc0N7zXpQLPnNOMyC0V3EI6vmZ8fG6ZoDXRDIy+5cwEcrNQsgOvKqf9Z17AR0Cc
A0g0P5zgFJsP9xipiLkk3VLUOnoUYUDhp3E6YD9QHgPdUsQ32t+BoySLXBRTOg3VU90Ux6fDETrK
eycaYG6PwSIrhQkKv+7oq9MrzZWmacHedtHa3MgdqL12zzaiBTxw6jARqnnLOF8TObvlGZxAh2d2
B0CdHkeMNdXo1kosik5/zVa8OYi1J9XMsbG4JFBwPsxJU/s1ROuw753OZgaDjfHwlugT8clqIWeS
4MpRvx34EldQrhzRgIKavh736CZnF/3DJ5SpEKjxezTvh1QBbVaBd6cUg6cHxFWreTxAnIksHwWA
5dqk29kTH3hO5tm8C8HbArVe/CfDNZalfn9CCyr5Xe95oM01msZO9df+b7+BGX/7DTur9P/cfbj/
qxd85Uxbm2nmoRtg4Xugq8Hz46CxJyeGKxtICz2jngmTusG1Q7PUduNAzdW87OODntRbBz7eS4AN
OAx206HL3hvkbjDf8exGfv60p8umtJ7sbgV1adqWBCHVYZuoo8CJpFwt2wLZJu6q215RL6zTYoHJ
R5c4lZLbUjY6U9DB6JTtHyx7X7pQXE7ppva/csbYhY0DeYELn7pf0DG1JsvluYDRvnsd8XHdwuem
WO89FH9uDkWjZB1YIcbQ2byMFvEaKsjHU+ka6WD8pVZz6P7JKhkcPjnAPRL18QIFAB93bWfgpooz
lW0kgkFUXBjW9vWyJ9+dLZEKwsSObMU7boHE+y31kC3yBh4aR7981tWj9MqljbyK+l+kRCfD/Zb/
TOkViSzMe1ekknVTdArfNNENBmhe97Ur9RjxtmmITY1ZVWKwb6FNDHLNRYymhmGKEHUFIdh2muDu
2k3u0nknPxteXw73t+fF9y9RM7pzR1C+G1h6XfLV/eF1ClD5nJJDhz8A7G+mCPctTFp0m3Lg0HSw
e1Y3iehSDe77i9N335+94zztBpoN33RfBqNOuObhvmKmu5/j4b54H4REty9qS+fD3SEiis9Va9qW
tiZzlOG/FF/ifqaP4O0YM/VpeDde4w7cN55Y9/rYjGeTUsSOtta0qy2oC7bh4T/rzIV9RwSpEfvF
L5crbpdpuu3O1ymCGzqSYOTvC5yRt5fEJ0TgAwmczc3rPPh8BR33tQ0KfEmKZrYHmKK+KTIPVS5h
YmFOTNDrkdByyJl4OjRTNxyrhFosKK2v6qq3vECITpBTltcoVTrLvcqyhTlY7ZzRdht/64PXTdsy
NVFyw25hUiO2s/cr82YvUTwUL66kssC9Gcj2STciHffDI2I3kUJkwNpyqRtgirRmKL1clWvAlRqg
aBSuCTZvXRNnpy4wNutFHFsXdg4hsjzb1h0omMWL4xK8u6F+HRkilM6rxUbVrN2VbF+qMUc6CjUM
BK4rIqscLV6Seabf9I48cydcXk/os3ktRkbtwORVmMQ+tXHOr32pHs+4IQDvdnG0KnLTdCAf2ffm
SN8MyopaMODrgj9pQlOLJPj4RZtEw7Zp533cV/tUTGVA3Fu6/8Ppy1f791Kau4fsATh6rwi2uqPf
TcMUvwelE+hh0dBLJugMNlrGpNVhS7/YZ8xohrXooBI17NQvIPgKKTIcN+D2dvvWHdX/Rh3JAfJn
qqht3Be4S9XiZFGbLoAR43X6C78jg39f/OKaLJgmeq2MjnBQcdtR7anHr0+4A6h7e1z7iM4GqmNu
p5uin2tweIgrdjn1q0OsNJrTADjZ7rA71kDr5gO5ZgmUKmf864P2e5+CofccP9FPv1jsHneQe/6E
cHOOTm/tSBFYtn3CiZiJNz1HsdQapfVKCK9nDzp+5NCok3EYMmk9swLv0nlehwtYScnrChxmcZKP
5LnSg/kYb+eRxOVdnCAz8vB7IpjzYOY27M0zsNMzZinr50o3+jDfCtw814LF2zawnDZk7K1RitK1
sKEMk9LhD2dt8J99fgQdK77EIRnn9Diu065f0nHzNmJ4eiNjFgEpM1ExiA+UGlYYtk6ZXra6Kqgu
5m2BJe4rpVTrd0s14HbmXdWQ+PKy7js0dq5HJt+Yx9s5+M+8OpESiltT6x898mCEkbwZOkOPUwUp
VQwX9IoaRP7+EUkH4n3scNMG9IwEx/xDAxBjiFdPXK8VcQSSSe4rF+lYxsOgP6cjxIe7Xz3DEBg0
vyTLgc0ktnG8eQFiYVBh8aOpIDd6UBY2ZURLtY3P4BFzlQY27dA1S3V2k1rTTe8PNkKeY9qmnJ5i
epgbUree+E3riED7sd3HmcDBeSSTwj7RLksOF2ysSy5/dGEP9Pe4FOPzOvKGET66KYEwwk5s5sHH
G6lcMn8ElNOF7dCiTpy73VEm7ugnRZ9ZY8YJ/n8eWtpgbfYowQZxraZuXsUjabqCnS12N8VTs8DG
CekO56RJDbOxZ9LDxVn0DfeKX99Gc1Hh0FTNMGnB2t2StwmLbyVvCaK31UTbdqb/521N/9/YmsPW
1uTk2ZIP7bRYEZ1uEjpksbYvEqWlo1hbyckLQ1QsEcxK8t+m5pjOQG0k03m5zYOdGIzvUzdzaqoz
nWjJkc/xaPjAKY7LofeG1RRp4TYLUiBnRbLuoSdbclz3oPQY+tajJgB3GD5+9vSxWaG4xWG+xKlC
LU5hMkv51Dg1y7snbOt0iVLnZ69+uDg7vyBn4CckoDx+adm1OKYeOabtnPOBe/+wPkYNn8i4djH5
4fZNeuGpecHvT/QtN3443hUbTyaRffmv7zXvAgQaElkOPG4niGB46pzc5jSnm/eBQTjy4FFaka3U
oPtmt72tafvWK96kGYkd2aOj6OaIPr2Togp9pNdxS+ErXKHAFQQ7UXHa1r2PDOIC+J+FNqcTcAwE
Hg1eskZUjellL/QmcLwt8SGMTfnCQYQesXM4Ny1jMB8UB5PtHNh0FTujm4fYs+VbZorK2DqLztns
nOc0aG8iSDyL04TEpTTJvkgRvB6Hljc3TkjX+YjWa6Pi0OFK51gnTsjzoZU228rbTeVgCjp+cnp3
mPNuTJsipPNKLcA9eg6fWYrDusQUh61zEfarnIzAV94dx78N3eMJcSiUbZ9KikPnW1sTQPix5Iib
1/BiTXLBo4hUAd5JKeum1xQi8KSCACmIYO//AFyQaiUQXwAA''')

import base64, gzip
_src = gzip.decompress(base64.b64decode(M50_OPENSET_B64))
with open('/kaggle/working/m50_openset.py', 'wb') as fh:
    fh.write(_src)
print('wrote m50_openset.py', len(_src), 'bytes')


## 4. Self-test --- runs before any real data

Builds a synthetic CirCor (two patients, four recordings) and pushes it through the real
index and the real scorers. It checks the things that fail *silently*: that the patient id is
parsed from the leading filename field so five locations of one child pool into one bootstrap
group rather than five, that the window loop never indexes past the end of a recording, that
no label is read, and that the AUROC is oriented so an unknown-is-high score gives ~1 --- a
scorer pointed the wrong way returns `1 - AUROC`, which looks like a result rather than a bug.

It also runs the M49 self-test, because the forward path and the gate come from that module.

In [ ]:
sys.path.insert(0, "/kaggle/working")
import m49_xval as X
import m50_openset as O

assert X.selftest() == 0, "M49 self-test failed - do not run the evaluation"
assert O.selftest() == 0, "M50 self-test failed - do not run the evaluation"


## 5. The gate, then the two questions

`m50_openset.run` re-scores the checkpoint on the 2,636 ICBHI test cycles first and asserts it
reproduces its own stored score. An open-set number from an unverified forward path is not a
result.

**Gate tolerance.** The default is `0.005`, not the `1e-3` M49 shipped with. M49 reproduced
0.5585 against a stored 0.5602 --- a gap of 0.0017, about 4 cycles in 2,636, caused by a
librosa version difference (M22_v2 was trained with librosa unpinned). 0.0017 is **8x below
the paper's own three-seed noise floor of 0.0141**, and a structurally wrong forward path
misses by 0.05--0.30, not 0.0017. The reproduced value is written into the results JSON next
to the stored one, so the gap is reported rather than hidden.

In [ ]:
doc = O.run(CIRCOR_ROOT, CKPT, WORK,
            icbhi_audio=ICBHI_AUDIO, icbhi_split=ICBHI_SPLIT,
            limit=SMOKE, batch_size=64, max_windows=MAX_WINDOWS)


## 6. Collect

Download `M50_results.zip` from the Output panel, unzip it into `M50_negative_control/`, and
commit.

**When you write this up:** it belongs in its own subsection, framed as trustworthiness, never
in the M49 transfer table. If it appears as "we also tested on CirCor" without the far-OOD
distinction, a reviewer who knows the corpus will read it as a category error.

In [ ]:
import shutil
shutil.make_archive("/kaggle/working/M50_results", "zip", WORK)
print("zipped:", os.path.getsize("/kaggle/working/M50_results.zip"), "bytes")

d = json.load(open(os.path.join(WORK, "results_M50_circor.json")))
o, c = d["openset"], d["confidence"]
print("\nenergy AUROC ", o["energy"]["auroc_unknown_vs_known"], o["energy"]["auroc_ci95"])
print("near-OOD ref ", o["near_ood_reference"]["auroc"],
      f"(n={o['near_ood_reference']['unknown_patients']} patients, HARDER task)")
print("confidence   ", c["circor"]["mean_max_softmax"], "on heart sounds vs",
      c["icbhi_test"]["mean_max_softmax"], "on ICBHI")
print("calls them   ", c["circor"]["predicted_class_fraction"])
